In [ ]:
import os
import shutil
from pathlib import Path
from dotenv import load_dotenv
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta import configure_spark_with_delta_pip


In [ ]:
STRING_COLS = {
    "usuarios":    ["nome", "email", "pais"],
    "planos":      ["nome"],
    "artistas":    ["nome", "pais"],
    "albuns":      ["titulo"],
    "musicas":     ["titulo"],
    "generos":     ["nome"],
    "playlists":   ["nome"],
    "dispositivos":["tipo", "so"],
}


In [ ]:
load_dotenv()

is_docker = os.path.exists("/.dockerenv")
MINIO_HOST = "minio:9000" if is_docker else "localhost:9000"
MINIO_USER = os.getenv("MINIO_ROOT_USER",     "minioadmin")
MINIO_PASS = os.getenv("MINIO_ROOT_PASSWORD", "minioadmin")

builder = (
    SparkSession.builder
    .appName("silver-dimensoes")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.defaultFS", "file:///")
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()

cliente_minio = Minio(
    MINIO_HOST, access_key=MINIO_USER, secret_key=MINIO_PASS, secure=False
)


In [ ]:
buckets = {b.name for b in cliente_minio.list_buckets()}
print("Buckets:", buckets)
if "silver" not in buckets:
    cliente_minio.make_bucket("silver")
    print("Bucket 'silver' criado.")


In [ ]:
def download_delta_do_minio(bucket: str, prefix: str,
                            cliente: Minio, local_dir: Path) -> None:
    if local_dir.exists():
        shutil.rmtree(local_dir)
    local_dir.mkdir(parents=True)
    objetos = list(cliente.list_objects(bucket, prefix=f"{prefix}/", recursive=True))
    print(f"  {len(objetos)} arquivo(s) em {bucket}/{prefix}/")
    for obj in objetos:
        relative = obj.object_name[len(prefix) + 1:]
        destino = local_dir / relative
        destino.parent.mkdir(parents=True, exist_ok=True)
        cliente.fget_object(bucket, obj.object_name, str(destino))


def upload_delta_para_minio(local_dir: Path, bucket: str, prefix: str,
                            cliente: Minio) -> None:
    for arq in sorted(local_dir.rglob("*")):
        if arq.is_file():
            object_name = f"{prefix}/{arq.relative_to(local_dir)}"
            cliente.fput_object(bucket, object_name, str(arq))
            print(f"  → {object_name}")


In [ ]:
for tabela, cols_str in STRING_COLS.items():
    print(f"\n{'=' * 55}")
    print(f"Silver: {tabela}")
    print(f"{'=' * 55}")

    local_bronze = Path(f"/tmp/bronze_local/{tabela}")
    download_delta_do_minio("bronze", f"dimensoes/{tabela}", cliente_minio, local_bronze)
    df = spark.read.format("delta").load(str(local_bronze))
    total_bronze = df.count()
    print(f"Bronze lido: {total_bronze:,} registros")

    df = df.dropDuplicates(["id"])
    removidos = total_bronze - df.count()
    if removidos:
        print(f"Duplicatas removidas: {removidos:,}")

    for col in cols_str:
        df = df.withColumn(col, F.trim(F.col(col)))

    local_silver = Path(f"/tmp/silver/{tabela}")
    if local_silver.exists():
        shutil.rmtree(local_silver)

    df.write.format("delta").mode("overwrite").save(str(local_silver))
    print(f"Silver Delta escrito em {local_silver}")

    print(f"Enviando para MinIO silver/dimensoes/{tabela}/")
    upload_delta_para_minio(local_silver, "silver", f"dimensoes/{tabela}", cliente_minio)
    print(f"{tabela} Silver concluído — {df.count():,} registros.")

print("\nTransformação Silver de todas as dimensões concluída.")


In [ ]:
spark.stop()
